In [2]:
import os
os.environ["HF_HOME"] = "D:/huggingface_cache"


In [3]:
from langchain_community.document_loaders import UnstructuredURLLoader
urls = ['https://www.livemint.com/economy/budget-2024-key-highlights-live-updates-nirmala-sitharaman-infrastructure-defence-income-tax-modi-budget-23-july-11721654502862.html',
        'https://cleartax.in/s/budget-2024-highlights',
        'https://www.hindustantimes.com/budget',
        'https://economictimes.indiatimes.com/news/economy/policy/budget-2024-highlights-india-nirmala-sitharaman-capex-fiscal-deficit-tax-slab-key-announcement-in-union-budget-2024-25/articleshow/111942707.cms?from=mdr']
loader = UnstructuredURLLoader(urls=urls)
data = loader.load()  


short text: "Access Denied". Defaulting to English.


In [ ]:
len(data)

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(data)


print("Total number of documents: ",len(docs))

Total number of documents:  177


In [ ]:
#docs[77]

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vectorstore = Chroma.from_documents(documents=docs, embedding=embedding_model)

In [7]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

retrieved_docs = retriever.invoke("Budget highlights")

In [ ]:
len(retrieved_docs)

In [8]:
print(retrieved_docs[2].page_content)

You Might Also Like:

You Might Also Like thumb-111943807

Budget 2o24: What's cheaper and what's costlier? Here's the list

Budget 2024 Announcements for STT, Short-term capital gains and LTCG


In [9]:
from langchain_huggingface import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from transformers import pipeline
from langchain_core.output_parsers import StrOutputParser

model_id = "distilgpt2"  # ⚠️ Much smaller model

text_generation_pipeline = pipeline(
    "text-generation",
    model=model_id,
    max_new_tokens=100,
    device=0
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
Answer the question using the context below:

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

C:\Users\Hamza Memon\.conda\envs\env_langchain\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hamza Memon\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [10]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain

In [13]:
question = "What is the Union Budget?"
response = rag_chain.invoke(question)

# Making the response readable
response = response.replace("</s>", "").strip()
print("Response:", response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Response: Answer the question using the context below:

Context:
[Document(id='63d8309b-14f4-4048-b8a1-636f49308304', metadata={'source': 'https://economictimes.indiatimes.com/news/economy/policy/budget-2024-highlights-india-nirmala-sitharaman-capex-fiscal-deficit-tax-slab-key-announcement-in-union-budget-2024-25/articleshow/111942707.cms?from=mdr'}, page_content='announcement for Women: Working women hostels and creche to be opened to facilitate higher participation of women in the workforce. ➤ Union Budget 2024 Highlights: Internship scheme for 1 crore youth in 500 top companies with Rs 5,000 per month as internship allowance and one-time assistance of Rs 6,000 to be provided. ➤ Budget 2024 announcement for Students: Financial support for loans up to Rs 10 lakh will be provided for higher education in domestic institutions. E-vouchers will be given directly to 1 lakh students every year for annual interest subvention of 3 per cent of the loan amount.'), Document(id='ee321ec5-a27c-458